# 02 Data checks and sample decisions

The audit in notebook 01 told me what the data contains. This notebook asks a different question: **is it fit for the analysis I'm about to run?**

The later sections use logarithms (for the Kaya decomposition), growth rates (for decoupling) and trade-adjusted emissions (for the offshoring question). Each of those can go wrong in a specific way, so I check for it here first. I also make two choices about the sample up front, rather than quietly deciding them halfway through the analysis.

**What's in here**

1. Check that every value is positive
2. Check that the emissions figures add up
3. Look at where the data jumps around, and why
4. See where the gaps in the data are
5. Decide on a minimum country size
6. Decide how to handle start and end years
7. Save the final analysis sample

In [ ]:
# --- Setup ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import (DATA_PROCESSED, FIGURES, TABLES, START_YEAR, END_YEAR,
                        CORE_COLUMNS, MIN_POPULATION, AVG_WINDOW)
from src.sample import analysis_sample, endpoint_means, classify_decoupling

plt.rcParams.update({"figure.figsize": (10, 5), "axes.spines.top": False, "axes.spines.right": False})

panel = pd.read_parquet(DATA_PROCESSED / "country_panel.parquet")
core = pd.read_parquet(DATA_PROCESSED / "core_panel.parquet")
print(f"Core panel: {core['country'].nunique()} countries, {START_YEAR}-{END_YEAR}")

## 1. Is every value positive?

The Kaya decomposition works with logarithms, and you can't take the log of zero or a negative number. One bad value would break the calculation for that country, so this needs to be a clean pass before going any further.

In [ ]:
checks = {col: int((core[col] <= 0).sum()) for col in CORE_COLUMNS}
missing = {col: int(core[col].isna().sum()) for col in CORE_COLUMNS}

pd.DataFrame({"zero_or_negative": checks, "missing": missing})

Both columns should be all zeros. If they are, the core panel is safe for the log-based methods later on.

## 2. Do the emissions figures add up?

Consumption-based emissions should equal production emissions plus the emissions embedded in trade:

**consumption = production + net traded emissions**

If that doesn't hold, the offshoring section would be built on inconsistent numbers. Checking it takes one line.

In [ ]:
trade = core.dropna(subset=["consumption_co2", "trade_co2"])
error = (trade["co2"] + trade["trade_co2"] - trade["consumption_co2"]).abs()

print(f"Rows checked: {len(trade):,}")
print(f"Largest mismatch: {error.max():.4f} million tonnes")

A mismatch this small is just rounding in the source file. The three columns are consistent with each other.

## 3. Where does the data jump around?

Growth rates are sensitive to sudden jumps. Here I flag any year where a country's emissions, GDP or energy use changed by more than 50% from the year before. The aim isn't to delete these rows. It's to understand them, because a jump can be a real event (a war, a collapse) or a problem with the data, and those need different handling.

In [ ]:
JUMP = 0.5  # 50% change from one year to the next

core_sorted = core.sort_values(["country", "year"])
flags = []
for col in ["co2", "gdp", "primary_energy_consumption"]:
    change = core_sorted.groupby("country")[col].pct_change()
    hit = core_sorted.loc[change.abs() > JUMP, ["country", "year"]].copy()
    hit["variable"] = col
    hit["change_pct"] = (change[change.abs() > JUMP] * 100).round(0)
    flags.append(hit)

jumps = pd.concat(flags).sort_values(["country", "year"]).reset_index(drop=True)
print(f"{len(jumps)} large jumps across {jumps['country'].nunique()} countries\n")

# Which countries have the most?
jumps["country"].value_counts().head(10)

In [ ]:
# Attach each country's population so we can see whether jumps are a small-country issue
pop_latest = core.loc[core["year"] == END_YEAR].set_index("country")["population"]
jumps["population_m"] = jumps["country"].map(pop_latest / 1e6).round(1)

print("Median population of countries with jumps:", round(jumps.drop_duplicates("country")["population_m"].median(), 1), "million")
print("Median population of all core countries:  ", round((pop_latest / 1e6).median(), 1), "million")

# A few well-known cases
jumps[jumps["country"].isin(["Kuwait", "Iraq", "Rwanda", "Liberia"])]

**What I take from this:** the jumps are mostly real events, not errors. Kuwait's emissions rose twelvefold in 1991 when its oil wells were set on fire during the Gulf War, then fell back the next year. Rwanda in 1994 and Liberia's civil wars all show up clearly. They also cluster in smaller, lower-income economies where one power station or one conflict moves the national total a lot.

So I'm not removing individual rows. Instead, the two decisions below (a minimum country size and averaged start and end points) reduce the influence these swings have on the results.

## 4. Where are the gaps?

A heatmap is the quickest way to see missing data across 200-odd countries and 35 years. Each row is a country and each column is a year. The colour shows how many of the five key variables are present (population, GDP, emissions, energy use, consumption emissions).

In [ ]:
key_vars = CORE_COLUMNS + ["consumption_co2"]
window = panel[panel["year"].between(START_YEAR, panel["year"].max())]

coverage = (window.assign(present=window[key_vars].notna().sum(axis=1))
                  .pivot(index="country", columns="year", values="present"))
coverage = coverage.loc[coverage.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(coverage.values, aspect="auto", cmap="Blues", vmin=0, vmax=len(key_vars))
ax.set_yticks([])
ax.set_ylabel(f"Countries ({len(coverage)}), best coverage at the top")
years = coverage.columns
step = 5
ax.set_xticks(range(0, len(years), step))
ax.set_xticklabels(years[::step])
ax.axvline(list(years).index(END_YEAR) + 0.5, color="black", linestyle="--", linewidth=1)
ax.text(list(years).index(END_YEAR) + 0.8, 3, "GDP data\nends here", fontsize=8, va="top")
ax.set_title("How complete is the data? (number of key variables present)")
fig.colorbar(im, ax=ax, label="Variables present (out of 5)", shrink=0.6)
fig.tight_layout()
fig.savefig(FIGURES / "02_data_coverage_heatmap.png", dpi=150)
plt.show()

Three things stand out. The top block is solid, which is the core panel. The lighter bands lower down are countries missing consumption-based emissions, energy data, or both. And the whole chart fades after 2022 because GDP stops there, which is exactly why the analysis window ends in 2022.

## 5. Decision: minimum country size

Very small countries can post huge percentage changes from tiny absolute amounts. A country with a few hundred thousand people opening one new plant could "double its emissions" and top a ranking, which would tell us nothing useful.

I'm setting a threshold of **1 million people** (in 2022). The cell below shows who that removes and how much of global emissions they account for. If the share is tiny, the threshold costs almost nothing and makes the rankings far more meaningful.

In [ ]:
sample = analysis_sample(core)
removed = sorted(set(core["country"]) - set(sample["country"]))

latest = core[core["year"] == END_YEAR].set_index("country")
removed_share = latest.loc[removed, "co2"].sum() / latest["co2"].sum() * 100

print(f"Threshold: {MIN_POPULATION:,} people")
print(f"Countries removed: {len(removed)}")
print(f"Their share of core-panel emissions in {END_YEAR}: {removed_share:.2f}%")
print(f"Analysis sample: {sample['country'].nunique()} countries\n")
latest.loc[removed, ["population", "co2"]].assign(population=lambda d: (d["population"] / 1e6).round(2)).rename(
    columns={"population": "population_m", "co2": "co2_mt"}).sort_values("population_m")

## 6. Decision: averaging the start and end points

Comparing 1990 with 2022 puts a lot of weight on two single years, and neither is a typical year. 1990 sits right next to the Gulf War and the break-up of the Soviet Union. 2020 to 2022 covers COVID and the energy price shock.

The fix is simple: compare the **average of 1990 to 1992** with the **average of 2020 to 2022**. To check it actually matters, I run the decoupling classification both ways and count how many countries change category.

In [ ]:
def classify(df, window):
    ends = endpoint_means(df, ["gdp", "co2"], window=window)
    return classify_decoupling(ends["gdp_start"], ends["gdp_end"], ends["co2_start"], ends["co2_end"])

single = classify(sample, window=1)
averaged = classify(sample, window=AVG_WINDOW)

comparison = pd.crosstab(single, averaged, rownames=["Single years"], colnames=[f"{AVG_WINDOW}-year averages"])
comparison

In [ ]:
switched = pd.DataFrame({"single_years": single, "averaged": averaged})
switched = switched[switched["single_years"] != switched["averaged"]]

print(f"{len(switched)} of {len(single)} countries change category depending on the method:\n")
switched

Most countries land in the same group either way, which is reassuring: the headline story doesn't depend on the method. The handful that do switch are the ones sitting close to a boundary, and those are exactly the cases where one odd year could tip the result. Using the averages means I can defend every classification if someone asks "what if you'd picked different years?".

From here on, all analysis uses the averaged start and end points.

## 7. Save the final analysis sample

Saving the sample means every later notebook starts from exactly the same set of countries.

In [ ]:
sample_table = (sample[sample["year"] == END_YEAR]
                .set_index("country")[["iso_code", "population"]]
                .assign(has_consumption_co2=sample.groupby("country")["consumption_co2"].apply(lambda s: s.notna().all())))
sample_table.to_csv(TABLES / "analysis_sample.csv")
sample.to_parquet(DATA_PROCESSED / "analysis_panel.parquet", index=False)

print(f"Saved {len(sample_table)} countries to outputs/tables/analysis_sample.csv")
print(f"Of these, {sample_table['has_consumption_co2'].sum()} have consumption-based emissions for every year")

## Summary

| Check | Result |
|---|---|
| All values positive | Pass, safe for log-based methods |
| Emissions add up | Pass, differences are rounding only |
| Large jumps | Real events, concentrated in small and conflict-affected countries |
| Data gaps | Core panel complete; GDP ends in 2022, which sets the window |
| Minimum size | 1 million people; removes a negligible share of emissions |
| Start and end points | 3-year averages; only a few borderline countries affected |

The analysis sample is saved and ready. Next up: how energy mixes have changed, and which countries have decoupled growth from emissions.